In [1]:

from __future__ import annotations
from dataclasses import dataclass
import math
from typing import Literal, Optional
import pynucastro as pyna
import pandas as pd
from pathlib import Path
from typing import Dict, List, Optional, Tuple


In [ ]:
"""
Alpha-decay rates from Sharma et al. (2021, https://doi.org/10.1016/j.nuclphysa.2021.122318) formulas, but computing l_min WITHOUT parity.

Paper provides (new modified) half-life formulae:
- NMHF  (Eq. 3)

We convert half-life -> decay constant: lambda = ln(2) / T_1/2

Source: Sharma et al., Nucl. Phys. A 1016 (2021) 122318
"""


LN2 = math.log(2.0)


# -----------------------------
# Coefficients from Table 1
# -----------------------------
# NMHF: a,b,c,d,e,f,g  (their Eq. 3 uses + g*l(l+1))
NMHF_COEF = dict(
    a=107.0131,
    b=-206.5398,
    c=-160.6152,
    d=309.6165,
    e=19.7237,
    f=-31.1655,
    g=0.0238,
)

# NMSF: a,b,c,d,e,f plus Ei by parity class (their Eq. 6 uses + f*l(l+1))
NMSF_COEF = dict(
    a=0.6975,
    b=-0.1113,
    c=-27.1154,
    d=13.5610,
    e=-24.7796,
    f=0.0174,
    Ei={"ee": 0.0000, "eo": -0.0396, "oe": -0.0350, "oo": 0.0438},
)

# NMMF: a,b,c,d,e,f  (their Eq. 9 uses + f*l(l+1))
NMMF_COEF = dict(
    a=-2.1876,
    b=20.5983,
    c=-74.2861,
    d=49.9026,
    e=-87.8618,
    f=0.0483,
)


# -----------------------------
# Helpers
# -----------------------------
def even_odd_class(Z: int, N: int) -> Literal["ee", "eo", "oe", "oo"]:
    """
    ee: even Z, even N
    eo: even Z, odd  N
    oe: odd  Z, even N
    oo: odd  Z, odd  N
    """
    z_even = (Z % 2 == 0)
    n_even = (N % 2 == 0)
    if z_even and n_even:
        return "ee"
    if z_even and (not n_even):
        return "eo"
    if (not z_even) and n_even:
        return "oe"
    return "oo"


def reduced_mass_amu(A_d: int, A_alpha: int = 4) -> float:
    """
    Paper uses reduced mass mu = A_d*A_alpha/(A_d + A_alpha) (dimensionless in amu units).
    """
    return (A_d * A_alpha) / (A_d + A_alpha)


def lmin_without_parity(j_parent: float, j_daughter: float) -> int:
    """
    REMOVE parity dependence: set lmin = |jp - jd| only.

    Note: jp and jd can be half-integers. |jp-jd| should be an integer for physical decays,
    but to be robust we round to nearest integer and verify it's close.
    """
    dj = abs(j_parent - j_daughter)
    l = int(round(dj))
   
    if l < 0:
        raise ValueError("Computed l < 0 (should never happen).")
    return l


@dataclass(frozen=True)
class AlphaDecayInput:
    """
    Parent -> Daughter + alpha.
    Provide Q_alpha in MeV, and parent/daughter spins (jp, jd).
    Parity is intentionally not included.
    """
    Z_parent: int
    A_parent: int
    Z_daughter: int
    A_daughter: int
    Q_alpha_MeV: float
    j_parent: float
    j_daughter: float


# -----------------------------
# Half-life formulae (log10 seconds)
# -----------------------------
def log10T_NMHF(x: AlphaDecayInput, l: Optional[int] = None) -> float:
    """
    Eq. (3) in the paper (NMHF), but you may provide l directly.
    If l is None, we compute l = |jp-jd| (no parity rule).
    """
    if x.Q_alpha_MeV <= 0:
        raise ValueError("Q_alpha must be > 0 MeV.")
    if l is None:
        l = lmin_without_parity(x.j_parent, x.j_daughter)

    mu = reduced_mass_amu(x.A_daughter, 4)
    Zd = x.Z_daughter
    Za = 2
    N_parent = x.A_parent - x.Z_parent
    I = (N_parent - x.Z_parent) / x.A_parent

    a, b, c, d, e, f, g = (NMHF_COEF[k] for k in ["a", "b", "c", "d", "e", "f", "g"])

    term1 = (a * math.sqrt(mu) + b) * (((Za * Zd) ** 0.6) * (x.Q_alpha_MeV ** (-0.5)) - 7.0)
    term2 = (c * math.sqrt(mu) + d) + e * I + f * (I ** 2)
    term3 = g * l * (l + 1)
    return term1 + term2 + term3

'''
def log10T_NMSF(x: AlphaDecayInput, l: Optional[int] = None) -> float:
    """
    Eq. (6) in the paper (NMSF), parity omitted only through l computation.
    Uses excitation energy Ei by even/odd class (Table 1).
    """
    if x.Q_alpha_MeV <= 0:
        raise ValueError("Q_alpha must be > 0 MeV.")
    if l is None:
        l = lmin_without_parity(x.j_parent, x.j_daughter)

    mu = reduced_mass_amu(x.A_daughter, 4)
    Zp = x.Z_parent
    N_parent = x.A_parent - x.Z_parent
    I = (N_parent - x.Z_parent) / x.A_parent

    cls = even_odd_class(x.Z_parent, N_parent)
    Ei = NMSF_COEF["Ei"][cls]

    a, b, c, d, e, f = (NMSF_COEF[k] for k in ["a", "b", "c", "d", "e", "f"])

    q_eff = x.Q_alpha_MeV - Ei
    if q_eff <= 0:
        return 'ERROR'
    else:
        term1 = a * Zp * math.sqrt(mu) * (q_eff ** (-0.5))
        term2 = b * Zp * math.sqrt(mu) + c + d * I + e * (I ** 2)
        term3 = f * l * (l + 1)
        return term1 + term2 + term3


def log10T_NMMF(x: AlphaDecayInput, l: Optional[int] = None) -> float:
    """
    Eq. (9) in the paper (NMMF), parity omitted only through l computation.
    """
    if x.Q_alpha_MeV <= 0:
        raise ValueError("Q_alpha must be > 0 MeV.")
    if l is None:
        l = lmin_without_parity(x.j_parent, x.j_daughter)

    mu = reduced_mass_amu(x.A_daughter, 4)
    Zd = x.Z_daughter
    N_parent = x.A_parent - x.Z_parent
    I = (N_parent - x.Z_parent) / x.A_parent

    a, b, c, d, e, f = (NMMF_COEF[k] for k in ["a", "b", "c", "d", "e", "f"])

    X = (Zd ** 0.4) / math.sqrt(x.Q_alpha_MeV)

    term_poly = a * math.sqrt(mu) * (X ** 2) + b * math.sqrt(mu) * X + c
    term_iso = d * I + e * (I ** 2)
    term_l = f * l * (l + 1)
    r= term_poly + term_iso + term_l
    if r>0:
        return r
    else:
        return 'ERROR'

'''

# -----------------------------
# Convert to rates
# -----------------------------
def half_life_seconds_from_log10(log10T: float) -> float:
    return 10.0 ** log10T

def decay_constant_from_half_life(T12_s: float) -> float:
    if T12_s <= 0:
        raise ValueError("Half-life must be > 0.")
    return LN2 / T12_s


def alpha_decay_rate(
    x: AlphaDecayInput,
    l: Optional[int] = None,
) -> dict:
    """
    Returns:
      - l_used
      - log10T_sec
      - T12_sec
      - lambda_1_per_s
    """
    
    log10T = log10T_NMHF(x, l=l)
    
    l_used = l if l is not None else lmin_without_parity(x.j_parent, x.j_daughter)
    if log10T!='ERROR':
        T12 = half_life_seconds_from_log10(log10T)
        lam = decay_constant_from_half_life(T12)
        return {"l_used": l_used, "log10T_sec": log10T, "T12_sec": T12, "lambda_1_per_s": lam}
    else:
        return {'lambda_1_per_s':'ERROR'}
    
def format_r1_block(parent,daughter,q_mev,jp,jd) -> str:
    """
    Writes a 3-line REACLIB-R1-like block:
      line1: parent daughter (padded) + source + Q
      line2: a0..a3
      line3: a4..a6
    For constant rate: rate(T9)=exp(a0), set a0=ln(lam), others 0.
    """
  
    decay=AlphaDecayInput(Z_parent=parent.Z,
                          A_parent=parent.A,
                          Z_daughter=daughter.Z,
                          A_daughter=daughter.A,
                          Q_alpha_MeV=q_mev,
                          j_parent=jp,
                          j_daughter=jd)


    
    lam=alpha_decay_rate(decay)['lambda_1_per_s']
    if lam != 'ERROR' and lam>0:
        a0 = math.log(lam)
        # line1: keep it whitespace-separated (parsers using split() will work)
        line1 = f"     {parent.short_spec_name:>5}  he4{daughter.short_spec_name:>5}"+" "*23+"wc12w    "+f"{q_mev: .5e}"+" "*10
        line2 = f"{a0: .6e}"+" 0.000000e+00 0.000000e+00 0.000000e+00                      "
        line3 = " 0.000000e+00 0.000000e+00 0.000000e+00                                   "

        return line1 + "\n" + line2 + "\n" + line3 + "\n"
    else:
        return 'ERROR'


   

def write_alpha_decay_file(
    out_path: str | Path,
    *,
    z_min: int = 50,
    z_max:int=118,
) -> None:
    """
    Create a file containing alpha- decay rates for all nuclides in WinVN.

    Inputs:
      - winvn_path: Winvn_v2.0-like file (must include Z, N, name, mass excess; stable flag helps δN)
      - out_path: output text file of 3-line blocks

    """
    nucs=pd.read_csv(r'winvne_v2.0.dat',skiprows=lambda x: not (x>7854 and (x-7855)%4==0),sep=r'\s+',names=['name','A','Z','N','spin','Mass excess (Mev)','source'])
   
    

    out_path = Path(out_path)
    n_written = 0
    n_skipped = 0
    by_ZN={}
    for i in range(len(nucs)):
        by_ZN[(nucs['Z'][i], nucs['N'][i])]=[nucs['name'][i],nucs['Mass excess (Mev)'][i],nucs['spin'][i]]

    with out_path.open("w", encoding="utf-8") as f:
        f.write("2"+" "*73+"\n")
        f.write(" "*74+"\n")
        f.write(" "*74+"\n")

        for i in range(len(nucs)):
            p=pyna.nucdata.nucleus.Nucleus(nucs['name'][i])
            if nucs['Z'][i] < z_min or nucs['Z'][i]>z_max:
                continue
            if p.tau=='stable':
                continue
            

            # Daughter: (Z+1, N-1)
            dkey = (nucs['Z'][i] -2, nucs['N'][i] -2)
            d = by_ZN.get(dkey)
            if d is None:
                n_skipped += 1
                continue

            # Qalpha from nuclear mass excess differences
            Qa = nucs["Mass excess (Mev)"][i]-d[1]-2.425
            if Qa <= 0.0:
                n_skipped += 1
                continue

            block = format_r1_block(p,pyna.nucdata.nucleus.Nucleus(d[0]),Qa,nucs['spin'][i],d[2])
            if block!='ERROR':
                f.write(block)
                n_written += 1
            else:
                n_skipped += 1

    print(f"Output: {out_path}")
    print(f"Written alpha- rates: {n_written}")
    print(f"Skipped (invalid log arg etc.): {n_skipped}")

write_alpha_decay_file('NMHF_alpha_R1')

C:\Users\Diego Hernandez\AppData\Local\Temp\ipykernel_1592\1146299249.py:201: RuntimeWarning: overflow encountered in scalar power
  return 10.0 ** log10T


Output: NMHF_alpha_R1
Written alpha- rates: 2933
Skipped (invalid log arg etc.): 2561
